<a href="https://colab.research.google.com/github/minsuk95lee-alt/violin-kinematics-audio-fusion/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Decoding the Adjudicator’s Eye: Multimodal Sensory Integration
# MSc Performance Science Pilot Study (N=14) - Data Analysis Pipeline
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.filterwarnings("ignore")

# (Note: Paths should be adjusted to the local or Drive environment prior to execution)
KINEMATICS_PATH = 'Master_Kinematics_Dataset.csv'
ACOUSTICS_PATH = 'Master_Acoustics_Dataset.csv'

try:
    df_k = pd.read_csv(KINEMATICS_PATH)
    df_a = pd.read_csv(ACOUSTICS_PATH)
    df_merged = pd.merge(df_k, df_a, on=['Subject', 'Label', 'Repertoire'])
    print(f"Data successfully loaded. Total multimodal samples: {len(df_merged)}")
except FileNotFoundError:
    print("Warning: CSV files not found in the current directory. Please verify the paths.")

# ==============================================================================
# 1. Statistical Analysis (Mann-Whitney U Test for Kinematic Jerk)
# ==============================================================================
print("\n--- 1. Kinematic Constraints by Repertoire ---")
# Focus on the Paganini repertoire constraint as hypothesized
if 'df_merged' in locals():
    df_paga = df_merged[df_merged['Repertoire'] == 'Paganini']
    if not df_paga.empty:
        winners_jerk = df_paga[df_paga['Label'] == 1]['R_Wrist_ISJ']
        non_winners_jerk = df_paga[df_paga['Label'] == 0]['R_Wrist_ISJ']
        stat, p = mannwhitneyu(winners_jerk, non_winners_jerk, alternative='two-sided')
        print(f"Winner Median Jerk: {winners_jerk.median():.0f}")
        print(f"Non-Winner Median Jerk: {non_winners_jerk.median():.0f}")
        print(f"Difference: Non-winners exhibit approx. 3x higher Kinematic Jerk.")

# ==============================================================================
# 2. Random Forest: Feature Importance (Visual Kinematics)
# ==============================================================================
print("\n--- 2. Feature Importance (Random Forest Baseline) ---")
visual_features = ['QoM_Total', 'Efficiency_Index', 'R_Wrist_ISJ', 'Kinematic_Entropy', 'Sync_Rate', 'Spatial_Occupancy_Std']

if 'df_merged' in locals():
    X_vis = df_merged[visual_features]
    y_target = df_merged['Label']

    rf_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
    rf_model.fit(X_vis, y_target)

    importance_df = pd.DataFrame({
        'Feature': visual_features,
        'Gini Importance': rf_model.feature_importances_
    }).sort_values(by='Gini Importance', ascending=False)
    print(importance_df.to_string(index=False))

# ==============================================================================
# 3. SVM Classification: Sight vs. Sound (Modality Comparison)
# ==============================================================================
print("\n--- 3. Modality Comparison (Linear SVM with LOOCV) ---")
audio_features = ['RMS_Mean', 'RMS_Std', 'Centroid_Mean', 'Centroid_Std',
                  'MFCC_1_Mean', 'MFCC_2_Mean', 'MFCC_3_Mean', 'MFCC_4_Mean']

def evaluate_modality(X, y, modality_name):
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    model = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1.0, random_state=42))

    for train_index, test_index in loo.split(X):
        model.fit(X[train_index], y[train_index])
        y_pred.append(model.predict(X[test_index])[0])
        y_true.append(y[test_index][0])

    acc = accuracy_score(y_true, y_pred) * 100
    print(f"{modality_name:20s} : Accuracy {acc:.1f}%")

if 'df_merged' in locals():
    X_audio = df_merged[audio_features].values
    X_fused = np.hstack((X_vis.values, X_audio))

    evaluate_modality(X_vis.values, y_target.values, "Visual-Only Model")
    evaluate_modality(X_audio, y_target.values, "Audio-Only Model")
    evaluate_modality(X_fused, y_target.values, "Fused Model (A+V)")